# Notebook 06: Time Series Analysis of the Gender Wage Gap

**EquiPay Canada — Temporal Dynamics & Trend Analysis**

---

## Overview

This notebook examines the **temporal evolution** of the gender wage gap in Canada from 2010 to 2025, incorporating macroeconomic context and advanced time series methodology.

## Key Research Questions

1. **Trend Analysis**: Has the wage gap been closing over time?
2. **Business Cycle Effects**: Does the gap widen during recessions (counter-cyclical)?
3. **Stationarity**: Is the gap trending toward zero (convergence)?
4. **Structural Breaks**: Did COVID-19 or policy changes alter the trajectory?
5. **Macro Relationships**: Phillips curve dynamics with wage gap?
6. **Forecasting**: What is the projected gap trajectory?

## Advanced Time Series Methods

### Unit Root & Stationarity Tests

| Test | Null Hypothesis | Implication |
|------|-----------------|-------------|
| **ADF** | Series has unit root | If rejected → stationary |
| **KPSS** | Series is stationary | If rejected → unit root |
| **Zivot-Andrews** | Unit root with break | Identifies structural break date |

### Cointegration Analysis
For wage gap and macro variables:
$$\text{Gap}_t = \alpha + \beta \cdot \text{Unemployment}_t + \epsilon_t$$

- **Engle-Granger Test**: Long-run equilibrium relationship
- **VECM**: Vector Error Correction Model for dynamics

### Structural Break Detection
- **Bai-Perron Test**: Multiple structural breaks
- **CUSUM Test**: Parameter stability
- **Chow Test**: Known break point (COVID-19)

### ARIMA & Forecasting
$$\phi(B)(1-B)^d Y_t = \theta(B)\epsilon_t$$

### Phillips Curve for Gender Gap
Testing whether the wage gap responds to labour market slack:
$$\text{Gap}_t = \alpha + \beta_1(\text{Unemployment}_t - \text{NAIRU}) + \beta_2 \Delta\text{CPI}_t + \epsilon_t$$

## Economic Hypotheses

| Hypothesis | Economic Theory | Test Method |
|------------|-----------------|-------------|
| Counter-cyclical gap | Recessions hit male-dominated sectors harder | Gap-Unemployment correlation |
| Inflation pass-through | Women have less bargaining power | Gap-CPI relationship |
| Policy effectiveness | Pay equity legislation works | Post-2018 structural break |
| Convergence | Gap approaches zero over time | ADF test on gap level |

## Data Source
- **Labour Force Survey PUMF** (Statistics Canada Catalogue 71M0001X)
- **Period**: 2010-2025 (16 annual observations)
- **Macroeconomic Data**: CPI, GDP growth, unemployment, interest rates

---

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

from src.constants import COLS, GENDER_CODES, normalize_column_names, DATA_SCOPE_START, DATA_SCOPE_END
from src.macro_data import MACRO_DATA, get_macro_dataframe, ECONOMIC_PERIODS

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})
plt.style.use('seaborn-v0_8-whitegrid')

# Ensure figures directory
Path('../reports/figures').mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("TIME SERIES ANALYSIS OF GENDER WAGE GAP")
print("=" * 70)
print(f"✓ Libraries loaded")
print(f"✓ Analysis period: {DATA_SCOPE_START}-{DATA_SCOPE_END}")
print("✓ Methods: Trend Analysis, ADF Test, ARIMA, Macro Correlation")

## 1. Data Loading & Preparation

In [ ]:
# Load LFS data
data_path = Path('../data/processed/lfs_processed.csv')
df = pd.read_csv(data_path)
df = normalize_column_names(df)

# Identify columns
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'
year_col = 'YEAR' if 'YEAR' in df.columns else 'SURVYEAR' if 'SURVYEAR' in df.columns else None

# Use REAL hourly earnings for time series analysis (CRITICAL for trend analysis)
# Without deflation, apparent wage "growth" is confounded with inflation
if COLS.REAL_HOURLY_EARNINGS in df.columns:
    wage_col = COLS.REAL_HOURLY_EARNINGS
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
elif 'REAL_HRLYEARN' in df.columns:
    wage_col = 'REAL_HRLYEARN'
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
else:
    wage_col = COLS.HOURLY_EARNINGS
    print("⚠ Real wages not available - trends will be confounded with inflation!")

# Create gender indicator
df['IS_FEMALE'] = (df[gender_col] == GENDER_CODES.get('Female', 2)).astype(int)

print(f"✓ Loaded {len(df):,} records")
print(f"  Years: {df[year_col].min()} - {df[year_col].max()}")

In [ ]:
# Load macro data
macro_df = get_macro_dataframe()
print("\nMacroeconomic Data:")
display(macro_df)

## 2. Calculate Annual Wage Gap Time Series

Aggregate LFS microdata to annual level and compute gender-specific wage statistics.

In [ ]:
# Calculate yearly wage statistics
yearly_stats = []

for year in sorted(df[year_col].unique()):
    year_data = df[df[year_col] == year]
    
    male_wages = year_data[year_data['IS_FEMALE'] == 0][wage_col]
    female_wages = year_data[year_data['IS_FEMALE'] == 1][wage_col]
    
    male_mean = male_wages.mean()
    female_mean = female_wages.mean()
    raw_gap = (male_mean - female_mean) / male_mean * 100 if male_mean > 0 else 0
    
    # Get macro data
    macro_year = macro_df[macro_df['year'] == year]
    cpi = macro_year['cpi'].values[0] if len(macro_year) > 0 else None
    gdp = macro_year['gdp_growth'].values[0] if len(macro_year) > 0 else None
    unemp = macro_year['unemployment'].values[0] if len(macro_year) > 0 else None
    
    yearly_stats.append({
        'year': int(year),
        'male_wage': male_mean,
        'female_wage': female_mean,
        'raw_gap_pct': raw_gap,
        'n_male': len(male_wages),
        'n_female': len(female_wages),
        'cpi': cpi,
        'gdp_growth': gdp,
        'unemployment': unemp
    })

ts_df = pd.DataFrame(yearly_stats)

# Calculate real wages (CPI-adjusted, base year 2010)
if ts_df['cpi'].notna().any():
    base_cpi = ts_df[ts_df['year'] == DATA_SCOPE_START]['cpi'].values[0]
    ts_df['real_male_wage'] = ts_df['male_wage'] * (base_cpi / ts_df['cpi'])
    ts_df['real_female_wage'] = ts_df['female_wage'] * (base_cpi / ts_df['cpi'])
    ts_df['real_gap_pct'] = (ts_df['real_male_wage'] - ts_df['real_female_wage']) / ts_df['real_male_wage'] * 100

print("Yearly Wage Statistics:")
display(ts_df.round(2))

## 3. Wage Trends Visualization

In [ ]:
# Plot: Nominal vs Real Wages by Gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nominal wages
ax1 = axes[0]
ax1.plot(ts_df['year'], ts_df['male_wage'], 'b-o', label='Male', linewidth=2)
ax1.plot(ts_df['year'], ts_df['female_wage'], 'r-s', label='Female', linewidth=2)
ax1.set_xlabel('Year')
ax1.set_ylabel('Hourly Wage ($)')
ax1.set_title('Nominal Wages by Gender')
ax1.legend()

# Mark economic periods
for period, (start, end) in ECONOMIC_PERIODS.items():
    if period == 'covid':
        ax1.axvspan(start, end, alpha=0.2, color='red', label='COVID-19')
    elif period == 'oil_shock':
        ax1.axvspan(start, end, alpha=0.2, color='gray', label='Oil Shock')

# Real wages (if available)
ax2 = axes[1]
if 'real_male_wage' in ts_df.columns:
    ax2.plot(ts_df['year'], ts_df['real_male_wage'], 'b-o', label='Male', linewidth=2)
    ax2.plot(ts_df['year'], ts_df['real_female_wage'], 'r-s', label='Female', linewidth=2)
    ax2.set_title('Real Wages by Gender (2010 dollars)')
else:
    ax2.plot(ts_df['year'], ts_df['male_wage'], 'b-o', label='Male', linewidth=2)
    ax2.plot(ts_df['year'], ts_df['female_wage'], 'r-s', label='Female', linewidth=2)
    ax2.set_title('Wages by Gender')

ax2.set_xlabel('Year')
ax2.set_ylabel('Hourly Wage ($)')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/figures/wage_trends_by_gender.png', dpi=150)
plt.show()

In [ ]:
# Plot: Gender Wage Gap Over Time
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(ts_df['year'], ts_df['raw_gap_pct'], 'o-', linewidth=2, color='purple', label='Raw Gap')

# Add trend line
z = np.polyfit(ts_df['year'], ts_df['raw_gap_pct'], 1)
p = np.poly1d(z)
ax.plot(ts_df['year'], p(ts_df['year']), '--', color='gray', alpha=0.7, label='Trend')

# Mark economic periods with vertical spans
covid_start, covid_end = ECONOMIC_PERIODS['covid']
ax.axvspan(covid_start, covid_end, alpha=0.2, color='red', label='COVID-19')

oil_start, oil_end = ECONOMIC_PERIODS['oil_shock']
ax.axvspan(oil_start, oil_end, alpha=0.2, color='gray', label='Oil Shock')

ax.set_xlabel('Year')
ax.set_ylabel('Gender Wage Gap (%)')
ax.set_title('Evolution of Gender Wage Gap in Canada (2010-2025)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add annotation
latest_gap = ts_df['raw_gap_pct'].iloc[-1]
earliest_gap = ts_df['raw_gap_pct'].iloc[0]
ax.annotate(f'{earliest_gap:.1f}%', (ts_df['year'].iloc[0], earliest_gap), 
            textcoords='offset points', xytext=(5, 5), fontsize=10)
ax.annotate(f'{latest_gap:.1f}%', (ts_df['year'].iloc[-1], latest_gap), 
            textcoords='offset points', xytext=(5, 5), fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/wage_gap_evolution.png', dpi=150)
plt.show()

# Trend interpretation
trend_slope = z[0]
print(f"\nTrend: {trend_slope:.3f} percentage points per year")
if trend_slope < 0:
    print(f"→ Gap is narrowing at ~{abs(trend_slope):.2f}pp per year")
else:
    print(f"→ Gap is widening at ~{trend_slope:.2f}pp per year")

## 4. Structural Break Analysis

Testing for structural breaks is critical for policy evaluation:
- **COVID-19 (2020)**: Economic shock with gendered impacts
- **Pay Equity Act (2018)**: Federal policy implementation
- **Oil Shock (2015-2016)**: Resource sector downturn

### Methods
1. **Chow Test**: Known break point
2. **Zivot-Andrews Test**: Endogenous break detection with unit root
3. **CUSUM Test**: Parameter stability over time

In [ ]:
# Analyze gap by economic period
print("="*60)
print("GENDER WAGE GAP BY ECONOMIC PERIOD")
print("="*60)

period_stats = []

for period_name, (start, end) in ECONOMIC_PERIODS.items():
    period_data = ts_df[(ts_df['year'] >= start) & (ts_df['year'] <= end)]
    
    if len(period_data) > 0:
        avg_gap = period_data['raw_gap_pct'].mean()
        avg_male = period_data['male_wage'].mean()
        avg_female = period_data['female_wage'].mean()
        
        period_stats.append({
            'Period': period_name.replace('_', ' ').title(),
            'Years': f"{start}-{end}",
            'Avg Gap (%)': avg_gap,
            'Male Wage': avg_male,
            'Female Wage': avg_female
        })
        
        print(f"\n{period_name.upper()} ({start}-{end}):")
        print(f"  Average gap: {avg_gap:.1f}%")
        print(f"  Male wage: ${avg_male:.2f}")
        print(f"  Female wage: ${avg_female:.2f}")

period_df = pd.DataFrame(period_stats)

In [ ]:
# ============================================================================
# STRUCTURAL BREAK ANALYSIS
# ============================================================================

print("=" * 70)
print("STRUCTURAL BREAK TESTS")
print("=" * 70)

# 1. Chow Test for COVID-19 Break (2020)
print("\n1. CHOW TEST FOR COVID-19 (2020)")
print("-" * 50)

# Split data at 2020
pre_covid = ts_df[ts_df['year'] < 2020]['raw_gap_pct']
post_covid = ts_df[ts_df['year'] >= 2020]['raw_gap_pct']

if len(pre_covid) > 2 and len(post_covid) > 2:
    # F-test for difference in distributions
    from scipy.stats import f_oneway, levene
    
    # Difference in means
    t_stat, p_mean = stats.ttest_ind(pre_covid, post_covid)
    
    # Difference in variance (Levene's test)
    _, p_var = levene(pre_covid, post_covid)
    
    print(f"\nPre-COVID ({2010}-2019):")
    print(f"  Mean gap: {pre_covid.mean():.2f}%")
    print(f"  Std dev:  {pre_covid.std():.2f}%")
    print(f"  n = {len(pre_covid)} years")
    
    print(f"\nPost-COVID (2020-{ts_df['year'].max()}):")
    print(f"  Mean gap: {post_covid.mean():.2f}%")
    print(f"  Std dev:  {post_covid.std():.2f}%")
    print(f"  n = {len(post_covid)} years")
    
    print(f"\nTest Results:")
    print(f"  Mean difference t-test: t = {t_stat:.3f}, p = {p_mean:.4f}")
    print(f"  Variance homogeneity (Levene): p = {p_var:.4f}")
    
    if p_mean < 0.05:
        print("\n  ✗ SIGNIFICANT structural break in mean at COVID-19")
    else:
        print("\n  ✓ No significant break in mean at COVID-19")

# 2. Zivot-Andrews Test (Endogenous Break Detection)
print("\n\n2. ZIVOT-ANDREWS TEST (Endogenous Break Detection)")
print("-" * 50)

try:
    from statsmodels.tsa.stattools import zivot_andrews
    
    gap_series = ts_df.set_index('year')['raw_gap_pct']
    
    # Run Zivot-Andrews test (tests for unit root with single break)
    za_result = zivot_andrews(gap_series, regression='c', autolag='AIC')
    
    print(f"\nZivot-Andrews Unit Root Test with Structural Break:")
    print(f"  Z-A statistic: {za_result[0]:.4f}")
    print(f"  Critical values: 1%: {za_result[2]['1%']:.3f}, 5%: {za_result[2]['5%']:.3f}")
    print(f"  Estimated break point: {za_result[1]}")
    
    if za_result[0] < za_result[2]['5%']:
        print(f"\n  → Reject unit root: Series is stationary around break at {za_result[1]}")
    else:
        print(f"\n  → Cannot reject unit root: Trend persistence even with break")
        
except Exception as e:
    print(f"  Zivot-Andrews test requires longer time series (n > 15)")
    print(f"  Current series length: {len(ts_df)}")

# 3. CUSUM Test for Parameter Stability
print("\n\n3. CUSUM TEST (Parameter Stability)")
print("-" * 50)

try:
    from statsmodels.stats.diagnostic import breaks_cusumolsresid
    
    # Fit simple trend model: gap = a + b*year
    X = sm.add_constant(ts_df['year'])
    model = sm.OLS(ts_df['raw_gap_pct'], X).fit()
    
    # CUSUM test on residuals
    cusum_stat, cusum_pval = breaks_cusumolsresid(model.resid)
    
    print(f"\nCUSUM test for trend model residuals:")
    print(f"  CUSUM statistic: {cusum_stat:.4f}")
    print(f"  p-value: {cusum_pval:.4f}")
    
    if cusum_pval < 0.05:
        print("  → Evidence of parameter instability (structural change)")
    else:
        print("  → Parameters appear stable over time")
        
except Exception as e:
    print(f"  CUSUM test: {str(e)[:50]}")

## 5. Macroeconomic Correlations

In [ ]:
# Correlation between wage gap and macro variables
print("="*60)
print("MACROECONOMIC CORRELATIONS")
print("="*60)

macro_vars = ['cpi', 'gdp_growth', 'unemployment']

for var in macro_vars:
    if ts_df[var].notna().sum() > 3:
        corr, pval = stats.pearsonr(
            ts_df['raw_gap_pct'].dropna(),
            ts_df[var].dropna()
        )
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f"{var.upper():<15} r = {corr:>6.3f} (p = {pval:.4f}) {sig}")

In [ ]:
# Visualization: Gap vs Macro Variables
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, var in enumerate(['cpi', 'gdp_growth', 'unemployment']):
    ax = axes[i]
    ax.scatter(ts_df[var], ts_df['raw_gap_pct'], alpha=0.7)
    
    # Trend line
    valid = ts_df[[var, 'raw_gap_pct']].dropna()
    if len(valid) > 2:
        z = np.polyfit(valid[var], valid['raw_gap_pct'], 1)
        p = np.poly1d(z)
        x_range = np.linspace(valid[var].min(), valid[var].max(), 100)
        ax.plot(x_range, p(x_range), 'r--', alpha=0.7)
    
    ax.set_xlabel(var.replace('_', ' ').title())
    ax.set_ylabel('Wage Gap (%)')
    ax.set_title(f'Wage Gap vs {var.upper()}')

plt.tight_layout()
plt.savefig('../reports/figures/gap_vs_macro.png', dpi=150)
plt.show()

## 5.1 Phillips Curve Analysis for Gender Wage Gap

The Phillips curve framework (Phillips, 1958; Blinder & Krueger, 1996) can be adapted to analyze how the wage gap responds to labour market conditions:

$$\text{Gap}_t = \alpha + \beta_1 (\text{Unemployment}_t - u^*) + \beta_2 \Delta\text{CPI}_t + \epsilon_t$$

**Hypotheses:**
- **Counter-cyclical gap**: β₁ > 0 (gap widens when unemployment rises, as male-dominated sectors contract)
- **Inflation pass-through**: β₂ ≠ 0 (differential bargaining power by gender)

In [ ]:
# ============================================================================
# PHILLIPS CURVE ANALYSIS FOR GENDER WAGE GAP
# ============================================================================

print("=" * 70)
print("PHILLIPS CURVE FRAMEWORK FOR WAGE GAP DYNAMICS")
print("=" * 70)

# Prepare variables
# Assume NAIRU around 6% for Canada
NAIRU = 6.0

ts_df['unemployment_gap'] = ts_df['unemployment'] - NAIRU
ts_df['inflation'] = ts_df['cpi'].pct_change() * 100  # YoY inflation proxy
ts_df['gap_change'] = ts_df['raw_gap_pct'].diff()

# Model 1: Simple Phillips relationship
# Gap = α + β(u - u*)
print("\n1. SIMPLE UNEMPLOYMENT-GAP RELATIONSHIP")
print("-" * 50)

valid_data = ts_df[['raw_gap_pct', 'unemployment_gap', 'unemployment']].dropna()

if len(valid_data) > 5:
    X = sm.add_constant(valid_data['unemployment_gap'])
    model_simple = sm.OLS(valid_data['raw_gap_pct'], X).fit()
    
    print(f"\nGap = α + β × (Unemployment - {NAIRU}%)")
    print(model_simple.summary().tables[1])
    
    beta_u = model_simple.params['unemployment_gap']
    se_u = model_simple.bse['unemployment_gap']
    
    print(f"\nInterpretation:")
    print(f"  β = {beta_u:.3f} (SE = {se_u:.3f})")
    if beta_u > 0:
        print(f"  → Counter-cyclical gap: 1pp rise in unemployment → {beta_u:.2f}pp wider gap")
    else:
        print(f"  → Pro-cyclical gap: 1pp rise in unemployment → {abs(beta_u):.2f}pp narrower gap")

# Model 2: Augmented with inflation
print("\n\n2. AUGMENTED PHILLIPS CURVE (with Inflation)")
print("-" * 50)

valid_aug = ts_df[['raw_gap_pct', 'unemployment_gap', 'inflation']].dropna()

if len(valid_aug) > 5:
    X_aug = sm.add_constant(valid_aug[['unemployment_gap', 'inflation']])
    model_aug = sm.OLS(valid_aug['raw_gap_pct'], X_aug).fit()
    
    print(f"\nGap = α + β₁×(U-U*) + β₂×Inflation")
    print(model_aug.summary().tables[1])
    
    print(f"\nR² = {model_aug.rsquared:.3f}")
    print(f"Adj. R² = {model_aug.rsquared_adj:.3f}")

# Visualization: Phillips Curve for Wage Gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Gap vs Unemployment
ax1 = axes[0]
scatter = ax1.scatter(ts_df['unemployment'], ts_df['raw_gap_pct'], 
                      c=ts_df['year'], cmap='viridis', s=80, edgecolor='black')
ax1.set_xlabel('Unemployment Rate (%)')
ax1.set_ylabel('Gender Wage Gap (%)')
ax1.set_title('Phillips Curve: Wage Gap vs Unemployment')
plt.colorbar(scatter, ax=ax1, label='Year')

# Add trend line
z = np.polyfit(ts_df['unemployment'].dropna(), 
               ts_df.loc[ts_df['unemployment'].notna(), 'raw_gap_pct'], 1)
p = np.poly1d(z)
x_line = np.linspace(ts_df['unemployment'].min(), ts_df['unemployment'].max(), 100)
ax1.plot(x_line, p(x_line), 'r--', label=f'Slope: {z[0]:.2f}')
ax1.axvline(NAIRU, color='gray', linestyle=':', alpha=0.7, label=f'NAIRU = {NAIRU}%')
ax1.legend()

# Right: Gap dynamics
ax2 = axes[1]
ax2.plot(ts_df['year'], ts_df['raw_gap_pct'], 'o-', color='purple', label='Wage Gap')
ax2_twin = ax2.twinx()
ax2_twin.plot(ts_df['year'], ts_df['unemployment'], 's--', color='orange', alpha=0.7, label='Unemployment')
ax2.set_xlabel('Year')
ax2.set_ylabel('Wage Gap (%)', color='purple')
ax2_twin.set_ylabel('Unemployment Rate (%)', color='orange')
ax2.set_title('Wage Gap and Unemployment Over Time')

# Combine legends
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../reports/figures/phillips_curve_wage_gap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print("ECONOMIC INTERPRETATION")
print("=" * 70)
print(f"""
The Phillips curve analysis tests whether the gender wage gap is counter-cyclical:

• Counter-cyclical gap (β > 0): During recessions, the gap WIDENS because:
  - Male-dominated sectors (construction, manufacturing) contract more
  - Women's concentration in services provides insulation

• Pro-cyclical gap (β < 0): During recessions, the gap NARROWS because:
  - Men's wages are more flexible downward
  - Women face 'sticky floor' effects

Canadian evidence from 2010-2025:
  The coefficient β = {model_simple.params['unemployment_gap']:.2f} suggests a 
  {'counter-cyclical' if model_simple.params['unemployment_gap'] > 0 else 'pro-cyclical'} pattern.
""")

## 5.2 Cointegration Analysis

Testing whether the wage gap and macroeconomic variables share a **long-run equilibrium relationship**.

If cointegrated:
- Short-run deviations are temporary
- System returns to equilibrium over time
- Implies policy should focus on long-run structural factors

### Engle-Granger Two-Step Method
1. Estimate long-run relationship: Gap_t = α + βX_t + ε_t
2. Test residuals for stationarity (ADF test)

In [ ]:
# ============================================================================
# COINTEGRATION ANALYSIS
# ============================================================================

print("=" * 70)
print("COINTEGRATION ANALYSIS: WAGE GAP & MACRO VARIABLES")
print("=" * 70)

from statsmodels.tsa.stattools import coint

def engle_granger_cointegration(y, x, var_name):
    """
    Engle-Granger two-step cointegration test.
    
    Step 1: Estimate long-run relationship y = α + βx + ε
    Step 2: Test ε for stationarity (ADF test)
    """
    # Align series
    data = pd.DataFrame({'y': y, 'x': x}).dropna()
    
    if len(data) < 8:
        return None
    
    # Step 1: Long-run regression
    X = sm.add_constant(data['x'])
    model = sm.OLS(data['y'], X).fit()
    residuals = model.resid
    
    # Step 2: ADF test on residuals
    # Use Engle-Granger critical values (more stringent)
    adf_result = adfuller(residuals, regression='c', autolag='AIC')
    
    # Engle-Granger 5% critical value for 2 variables is approximately -3.34
    eg_critical = -3.34
    
    return {
        'var_name': var_name,
        'beta': model.params.iloc[1],
        'beta_se': model.bse.iloc[1],
        't_stat': model.tvalues.iloc[1],
        'r_squared': model.rsquared,
        'adf_stat': adf_result[0],
        'adf_pval': adf_result[1],
        'eg_critical': eg_critical,
        'cointegrated': adf_result[0] < eg_critical
    }

# Test cointegration with each macro variable
print("\nEngle-Granger Cointegration Tests:")
print("(H₀: No cointegration, requires ADF stat < critical value to reject)")
print("-" * 70)

gap_series = ts_df.set_index('year')['raw_gap_pct']

coint_results = []
for var in ['unemployment', 'cpi', 'gdp_growth']:
    macro_series = ts_df.set_index('year')[var]
    result = engle_granger_cointegration(gap_series, macro_series, var)
    
    if result:
        coint_results.append(result)
        coint_str = "Yes ✓" if result['cointegrated'] else "No"
        print(f"\n{var.upper()}:")
        print(f"  Long-run β: {result['beta']:.3f} (SE = {result['beta_se']:.3f})")
        print(f"  R²: {result['r_squared']:.3f}")
        print(f"  Residual ADF: {result['adf_stat']:.3f} (5% critical: {result['eg_critical']:.3f})")
        print(f"  Cointegrated: {coint_str}")

# Johansen test for multivariate cointegration (if enough data)
print("\n\n" + "=" * 70)
print("JOHANSEN COINTEGRATION TEST (Multivariate)")
print("=" * 70)

try:
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    
    # Prepare multivariate system
    system_vars = ['raw_gap_pct', 'unemployment', 'cpi']
    system_data = ts_df[system_vars].dropna()
    
    if len(system_data) >= 12:
        johansen_result = coint_johansen(system_data, det_order=0, k_ar_diff=1)
        
        print(f"\nVariables: Wage Gap, Unemployment, CPI")
        print(f"Sample size: {len(system_data)} years")
        
        print(f"\n{'# Coint. Vectors':<20} {'Trace Stat':<15} {'5% Critical':<15} {'Reject H₀?'}")
        print("-" * 60)
        
        for i in range(len(system_vars)):
            trace = johansen_result.lr1[i]
            crit = johansen_result.cvt[i, 1]  # 5% critical value
            reject = "Yes" if trace > crit else "No"
            print(f"r = {i:<18} {trace:<15.2f} {crit:<15.2f} {reject}")
        
        # Count cointegrating vectors
        n_coint = sum(johansen_result.lr1 > johansen_result.cvt[:, 1])
        print(f"\n→ Number of cointegrating relationships: {n_coint}")
        
    else:
        print("Insufficient data for Johansen test (need n ≥ 12)")
        
except Exception as e:
    print(f"Johansen test error: {str(e)[:60]}")

print("\n" + "=" * 70)
print("COINTEGRATION INTERPRETATION")
print("=" * 70)
print("""
If wage gap is cointegrated with unemployment:
  → Long-run equilibrium exists between gap and labour market conditions
  → Policy can exploit this relationship for targeting

If NOT cointegrated:
  → Wage gap evolves independently of business cycle
  → Structural policies (not stabilization) are needed
""")

## 5.3 Granger Causality Tests

Testing **predictive causality** between wage gap and macroeconomic variables:

$$\text{Gap}_t = \sum_{i=1}^{p} \alpha_i \text{Gap}_{t-i} + \sum_{j=1}^{p} \beta_j X_{t-j} + \epsilon_t$$

- **H₀**: X does not Granger-cause Gap (β₁ = β₂ = ... = 0)
- If rejected: Past values of X help predict Gap

This is **predictive**, not causal in the structural sense (Granger, 1969).

In [ ]:
# ============================================================================
# GRANGER CAUSALITY TESTS
# ============================================================================

print("=" * 70)
print("GRANGER CAUSALITY TESTS")
print("=" * 70)

from statsmodels.tsa.stattools import grangercausalitytests

def run_granger_test(y, x, max_lag=2, verbose=False):
    """
    Run Granger causality test: Does x Granger-cause y?
    """
    data = pd.DataFrame({'y': y, 'x': x}).dropna()
    
    if len(data) < 10:
        return None
    
    try:
        result = grangercausalitytests(data[['y', 'x']], maxlag=max_lag, verbose=verbose)
        
        # Extract F-test results for each lag
        results_summary = {}
        for lag in range(1, max_lag + 1):
            f_stat = result[lag][0]['ssr_ftest'][0]
            p_val = result[lag][0]['ssr_ftest'][1]
            results_summary[lag] = {'f_stat': f_stat, 'p_val': p_val}
        
        return results_summary
    except Exception as e:
        return None

# Test each direction
print("\nTesting predictive relationships (max lag = 2 years):")
print("-" * 70)

gap_series = ts_df['raw_gap_pct']

# 1. Does Unemployment Granger-cause Gap?
print("\n1. Does UNEMPLOYMENT Granger-cause WAGE GAP?")
gc_unemp_gap = run_granger_test(gap_series, ts_df['unemployment'], max_lag=2)
if gc_unemp_gap:
    for lag, res in gc_unemp_gap.items():
        sig = "**" if res['p_val'] < 0.05 else ""
        print(f"   Lag {lag}: F = {res['f_stat']:.2f}, p = {res['p_val']:.4f} {sig}")
else:
    print("   Insufficient data")

# 2. Does Gap Granger-cause Unemployment?
print("\n2. Does WAGE GAP Granger-cause UNEMPLOYMENT?")
gc_gap_unemp = run_granger_test(ts_df['unemployment'], gap_series, max_lag=2)
if gc_gap_unemp:
    for lag, res in gc_gap_unemp.items():
        sig = "**" if res['p_val'] < 0.05 else ""
        print(f"   Lag {lag}: F = {res['f_stat']:.2f}, p = {res['p_val']:.4f} {sig}")
else:
    print("   Insufficient data")

# 3. Does GDP growth Granger-cause Gap?
print("\n3. Does GDP GROWTH Granger-cause WAGE GAP?")
gc_gdp_gap = run_granger_test(gap_series, ts_df['gdp_growth'], max_lag=2)
if gc_gdp_gap:
    for lag, res in gc_gdp_gap.items():
        sig = "**" if res['p_val'] < 0.05 else ""
        print(f"   Lag {lag}: F = {res['f_stat']:.2f}, p = {res['p_val']:.4f} {sig}")
else:
    print("   Insufficient data")

# 4. Does CPI Granger-cause Gap?
print("\n4. Does CPI Granger-cause WAGE GAP?")
gc_cpi_gap = run_granger_test(gap_series, ts_df['cpi'], max_lag=2)
if gc_cpi_gap:
    for lag, res in gc_cpi_gap.items():
        sig = "**" if res['p_val'] < 0.05 else ""
        print(f"   Lag {lag}: F = {res['f_stat']:.2f}, p = {res['p_val']:.4f} {sig}")
else:
    print("   Insufficient data")

print("\n" + "=" * 70)
print("GRANGER CAUSALITY INTERPRETATION")
print("=" * 70)
print("""
Interpretation of results:
• ** indicates p < 0.05 (significant Granger-causality)

If Unemployment → Gap:
  Past unemployment helps predict current wage gap
  → Business cycle effects on gender inequality

If Gap → Unemployment:
  Past wage gap helps predict current unemployment
  → Gender inequality affects aggregate labour market

CAUTION: Granger causality is predictive, not structural causality.
With only 16 annual observations, results should be interpreted cautiously.
""")

## 6. Time Series Decomposition

In [ ]:
# Decompose wage gap time series
if len(ts_df) >= 6:  # Need at least 2 periods for decomposition
    # Create a simple time series
    gap_series = pd.Series(
        ts_df['raw_gap_pct'].values,
        index=pd.date_range(start=str(ts_df['year'].min()), periods=len(ts_df), freq='YE')
    )
    
    # Moving average for trend
    gap_series_ma = gap_series.rolling(window=3, center=True).mean()
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Original + Trend
    axes[0].plot(gap_series.index, gap_series.values, 'b-o', label='Actual', alpha=0.7)
    axes[0].plot(gap_series.index, gap_series_ma.values, 'r-', label='3-Year Moving Average', linewidth=2)
    axes[0].set_ylabel('Wage Gap (%)')
    axes[0].set_title('Gender Wage Gap: Original and Trend')
    axes[0].legend()
    
    # Deviations from trend
    deviations = gap_series - gap_series_ma
    axes[1].bar(gap_series.index, deviations.values, color='steelblue', alpha=0.7)
    axes[1].axhline(0, color='black', linewidth=0.5)
    axes[1].set_ylabel('Deviation from Trend (pp)')
    axes[1].set_title('Cyclical Component')
    
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for time series decomposition")

## 7. Regression Analysis with Time Trend

In [ ]:
# Time trend regression
print("="*60)
print("TIME TREND REGRESSION")
print("="*60)

# Create time trend variable
ts_df['time_trend'] = ts_df['year'] - ts_df['year'].min()

# Basic model: Gap = α + β·Time + ε
X_basic = sm.add_constant(ts_df['time_trend'])
y = ts_df['raw_gap_pct']

model_basic = sm.OLS(y, X_basic).fit()

print("\nModel 1: Gap = α + β·Time")
print(f"  Intercept: {model_basic.params['const']:.3f}")
print(f"  Time trend: {model_basic.params['time_trend']:.4f} pp/year")
print(f"  R²: {model_basic.rsquared:.4f}")

# Extended model with macro controls
ts_clean = ts_df[['raw_gap_pct', 'time_trend', 'gdp_growth', 'unemployment']].dropna()
if len(ts_clean) >= 5:
    X_extended = sm.add_constant(ts_clean[['time_trend', 'gdp_growth', 'unemployment']])
    y_ext = ts_clean['raw_gap_pct']
    
    model_ext = sm.OLS(y_ext, X_extended).fit()
    
    print("\nModel 2: Gap = α + β₁·Time + β₂·GDP + β₃·Unemployment")
    print(f"  Time trend: {model_ext.params['time_trend']:.4f} pp/year")
    print(f"  GDP effect: {model_ext.params['gdp_growth']:.4f}")
    print(f"  Unemployment effect: {model_ext.params['unemployment']:.4f}")
    print(f"  R²: {model_ext.rsquared:.4f}")

## 8. Projection (Simple Forecast)

In [ ]:
# Project wage gap into future
print("="*60)
print("WAGE GAP PROJECTION (2026-2030)")
print("="*60)

future_years = np.arange(2026, 2031)
future_trend = future_years - ts_df['year'].min()

# Predict using linear model
X_future = sm.add_constant(pd.DataFrame({'time_trend': future_trend}))
predictions = model_basic.predict(X_future)

# Confidence intervals (approximate)
se = np.sqrt(model_basic.mse_resid)
ci_lower = predictions - 1.96 * se
ci_upper = predictions + 1.96 * se

print(f"\n{'Year':<8} {'Predicted Gap':>15} {'95% CI':>20}")
print("-" * 45)
for year, pred, low, high in zip(future_years, predictions, ci_lower, ci_upper):
    print(f"{year:<8} {pred:>14.1f}% [{low:.1f}% - {high:.1f}%]")

# When will gap close?
if model_basic.params['time_trend'] < 0:
    years_to_close = model_basic.params['const'] / abs(model_basic.params['time_trend'])
    close_year = int(ts_df['year'].min() + years_to_close)
    print(f"\n→ At current rate, gap projected to close by: {close_year}")
else:
    print(f"\n→ Gap is widening, not projected to close")

In [ ]:
# Visualization: Historical + Projection
fig, ax = plt.subplots(figsize=(12, 6))

# Historical
ax.plot(ts_df['year'], ts_df['raw_gap_pct'], 'bo-', linewidth=2, markersize=8, label='Historical')

# Trend line (full range)
all_years = np.concatenate([ts_df['year'].values, future_years])
all_trend = all_years - ts_df['year'].min()
X_all = sm.add_constant(pd.DataFrame({'time_trend': all_trend}))
all_pred = model_basic.predict(X_all)
ax.plot(all_years, all_pred, '--', color='gray', alpha=0.7, label='Trend')

# Future predictions
ax.plot(future_years, predictions, 'rs-', linewidth=2, markersize=8, label='Projection')
ax.fill_between(future_years, ci_lower, ci_upper, alpha=0.2, color='red')

# Vertical line at present
ax.axvline(ts_df['year'].max(), color='green', linestyle=':', alpha=0.7, label='Present')

ax.set_xlabel('Year')
ax.set_ylabel('Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap: Historical Trends and Projection')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/wage_gap_projection.png', dpi=150)
plt.show()

## 9. Summary and Export

In [ ]:
# Summary
print("\n" + "="*70)
print("TIME SERIES ANALYSIS SUMMARY")
print("="*70)

earliest_gap = ts_df['raw_gap_pct'].iloc[0]
latest_gap = ts_df['raw_gap_pct'].iloc[-1]
change = latest_gap - earliest_gap

print(f"""
PERIOD: {ts_df['year'].min()} - {ts_df['year'].max()}

WAGE GAP EVOLUTION:
  Starting gap ({ts_df['year'].min()}): {earliest_gap:.1f}%
  Current gap ({ts_df['year'].max()}): {latest_gap:.1f}%
  Total change: {change:+.1f} percentage points

TREND ANALYSIS:
  Annual change: {model_basic.params['time_trend']:.3f}pp per year
  Direction: {'Narrowing ↓' if model_basic.params['time_trend'] < 0 else 'Widening ↑'}
  R²: {model_basic.rsquared:.3f}

KEY FINDINGS:
""")

# COVID impact
if len(pre_covid) > 0 and len(post_covid) > 0:
    covid_effect = post_covid['raw_gap_pct'].mean() - pre_covid['raw_gap_pct'].mean()
    print(f"  COVID-19 Impact: {covid_effect:+.1f}pp {'(gap widened)' if covid_effect > 0 else '(gap narrowed)'}")

# Macro correlations
print(f"")
for var in ['unemployment', 'gdp_growth']:
    if ts_df[var].notna().sum() > 3:
        corr, _ = stats.pearsonr(ts_df['raw_gap_pct'].dropna(), ts_df[var].dropna())
        relation = 'positively' if corr > 0 else 'negatively'
        print(f"  {var.replace('_', ' ').title()}: {relation} correlated (r = {corr:.2f})")

In [ ]:
# Save results
results_path = Path('../reports')
results_path.mkdir(exist_ok=True)

# Save time series data
ts_df.to_csv(results_path / 'wage_gap_timeseries.csv', index=False)

# Save period analysis
period_df.to_csv(results_path / 'wage_gap_by_period.csv', index=False)

print(f"\n✓ Results saved to {results_path}")
print(f"  - wage_gap_timeseries.csv")
print(f"  - wage_gap_by_period.csv")

## 1. Load and Prepare Time Series Data

In [ ]:
# Load REAL Time Series Data from Statistics Canada
# Source: Table 14-10-0064-01 (Wages by sex and age)
# Vector IDs: v2133719 (Men+), v2134859 (Women+)
# Data: 2010-2025 yearly averages of monthly hourly wages

import requests

# Load from pre-fetched file (REAL Statistics Canada data)
yearly_wages_file = Path('../data/processed/yearly_wages_by_gender.csv')

if yearly_wages_file.exists():
    ts_df = pd.read_csv(yearly_wages_file)
    print(f"✅ Loaded REAL yearly wage data from Statistics Canada")
    print(f"   File: {yearly_wages_file}")
    print(f"   Source: {ts_df['source'].iloc[0]}")
    print(f"   Years: {ts_df['year'].min()}-{ts_df['year'].max()} ({len(ts_data)} observations)")
else:
    # Fetch from API if file doesn't exist
    print("Fetching REAL data from Statistics Canada Vector API...")
    url = 'https://www150.statcan.gc.ca/t1/wds/rest/getDataFromVectorsAndLatestNPeriods'
    payload = [
        {'vectorId': 2133719, 'latestN': 200},  # Men+, 15+ years, Avg hourly wage
        {'vectorId': 2134859, 'latestN': 200}   # Women+, 15+ years, Avg hourly wage
    ]
    
    resp = requests.post(url, json=payload, timeout=60)
    data = resp.json()
    
    if all(d['status'] == 'SUCCESS' for d in data):
        male_points = data[0]['object']['vectorDataPoint']
        female_points = data[1]['object']['vectorDataPoint']
        
        # Aggregate to yearly
        male_by_year, female_by_year = {}, {}
        for p in male_points:
            year = int(p['refPer'][:4])
            if 2010 <= year <= 2025 and p['value']:
                male_by_year.setdefault(year, []).append(float(p['value']))
        for p in female_points:
            year = int(p['refPer'][:4])
            if 2010 <= year <= 2025 and p['value']:
                female_by_year.setdefault(year, []).append(float(p['value']))
        
        records = []
        for year in sorted(set(male_by_year.keys()) & set(female_by_year.keys())):
            m = sum(male_by_year[year]) / len(male_by_year[year])
            f = sum(female_by_year[year]) / len(female_by_year[year])
            records.append({
                'year': year, 'male_wage': round(m, 2), 'female_wage': round(f, 2),
                'wage_gap': round((m - f) / m * 100, 2), 'wage_ratio': round(f / m, 4),
                'source': 'StatCan_API_14-10-0064-01'
            })
        
        ts_df = pd.DataFrame(records)
        ts_df.to_csv(yearly_wages_file, index=False)
        print(f"✅ Fetched and saved {len(ts_data)} years of REAL wage data")
    else:
        raise ValueError("Failed to fetch data from Statistics Canada API")

# Add macroeconomic controls from centralized module
if MACRO_AVAILABLE:
    macro_df = get_macro_dataframe()
    ts_df = ts_df.merge(
        macro_df[['year', 'cpi', 'gdp_growth', 'unemployment', 'interest_rate', 'inflation', 'recession', 'covid']],
        on='year', how='left'
    )
    print("✅ Added macroeconomic controls from centralized module")
else:
    # Add macro variables inline using the centralized data
    from macro_data import MACRO_DATA
    for var in ['cpi', 'gdp_growth', 'unemployment', 'interest_rate']:
        ts_df[var] = ts_df['year'].map(lambda y, v=var: MACRO_DATA.get(y, {}).get(v, np.nan))
    ts_df['inflation'] = ts_df['cpi'].pct_change() * 100
    ts_df['recession'] = (ts_df['gdp_growth'] < 0).astype(int)
    ts_df['covid'] = ts_df['year'].isin([2020, 2021]).astype(int)
    print("✅ Added macroeconomic controls")

print(f"\n📅 Data Period: {ts_df['year'].min()} - {ts_df['year'].max()}")
print(f"📊 Source: REAL Statistics Canada Data (Table 14-10-0064-01)")
print(f"📈 Observations: {len(ts_data)} years of REAL wage data (NO simulation)")

In [ ]:
# Prepare time series with macroeconomic controls and real wage calculations

# Ensure wage gap and ratio are calculated (may already exist from loaded data)
if 'wage_gap' not in ts_df.columns:
    ts_df['wage_gap'] = (ts_df['male_wage'] - ts_df['female_wage']) / ts_df['male_wage'] * 100
if 'wage_ratio' not in ts_df.columns:
    ts_df['wage_ratio'] = ts_df['female_wage'] / ts_df['male_wage']

# Get base CPI for real wage calculations (2010 = base year)
if MACRO_AVAILABLE:
    base_cpi = MACRO_DATA[BASE_YEAR]['cpi']
else:
    base_cpi = ts_df.loc[ts_df['year'] == 2010, 'cpi'].values[0] if 'cpi' in ts_df.columns else 116.5

# Calculate real wages (inflation-adjusted to 2010 dollars)
if 'cpi' in ts_df.columns:
    ts_df['real_male_wage'] = ts_df['male_wage'] * (base_cpi / ts_df['cpi'])
    ts_df['real_female_wage'] = ts_df['female_wage'] * (base_cpi / ts_df['cpi'])
    ts_df['real_wage_gap'] = (ts_df['real_male_wage'] - ts_df['real_female_wage']) / ts_df['real_male_wage'] * 100
else:
    # Use nominal values if CPI not available
    ts_df['real_male_wage'] = ts_df['male_wage']
    ts_df['real_female_wage'] = ts_df['female_wage']
    ts_df['real_wage_gap'] = ts_df['wage_gap']

# Add trend variable for regression analysis
ts_df['trend'] = ts_df['year'] - 2010

# Display the data
print("📊 REAL Time Series Data with Macro Controls (2010-2025)")
print("=" * 90)
display_cols = ['year', 'male_wage', 'female_wage', 'wage_gap', 'wage_ratio']
if 'cpi' in ts_df.columns:
    display_cols += ['real_male_wage', 'real_female_wage', 'real_wage_gap']
if 'gdp_growth' in ts_df.columns:
    display_cols += ['gdp_growth', 'unemployment']
display(ts_df[display_cols].round(2))
print("=" * 90)

# Summary statistics
first_year = ts_df['year'].iloc[0]
last_year = ts_df['year'].iloc[-1]
first_gap = ts_df['wage_gap'].iloc[0]
last_gap = ts_df['wage_gap'].iloc[-1]
gap_change = last_gap - first_gap

print(f"\n📈 REAL Statistics Canada Data Summary:")
print(f"   Wage gap {first_year}: {first_gap:.2f}%")
print(f"   Wage gap {last_year}: {last_gap:.2f}%")
print(f"   Gap change {first_year}→{last_year}: {gap_change:+.2f} percentage points")
print(f"   Relative improvement: {abs(gap_change)/first_gap*100:.1f}%")
if 'real_wage_gap' in ts_df.columns:
    print(f"   Real wage gap (2010$) {last_year}: {ts_df['real_wage_gap'].iloc[-1]:.2f}%")

In [ ]:
# Initialize Time Series Analyzer with REAL Statistics Canada data
# Since our data is already aggregated (yearly male_wage, female_wage, wage_gap),
# we create a wrapper that directly uses ts_data

class YearlyWageAnalyzer:
    """
    Time series analyzer for pre-aggregated yearly wage data.
    Provides all analysis methods from WageGapTimeSeriesAnalyzer.
    """
    
    def __init__(self, data: pd.DataFrame):
        self.data = data.copy()
        self.ts_data = data.copy()  # Already in correct format
        
    def trend_analysis(self):
        """Analyze linear and quadratic trends in the wage gap."""
        import statsmodels.api as sm
        from statsmodels.regression.linear_model import OLS
        
        series = self.ts_df['wage_gap'].dropna()
        years = self.ts_df['year'].values[:len(series)]
        
        results = {}
        
        # Linear trend
        X = sm.add_constant(years)
        trend_model = OLS(series, X).fit()
        
        results['linear_trend'] = {
            'intercept': trend_model.params[0],
            'slope': trend_model.params[1],
            'slope_interpretation': f"Wage gap changes by {trend_model.params[1]:.3f} pp per year",
            'r_squared': trend_model.rsquared,
            'p_value_slope': trend_model.pvalues[1],
            'significant_trend': trend_model.pvalues[1] < 0.05
        }
        
        # Quadratic trend
        X_quad = np.column_stack([np.ones(len(years)), years, years**2])
        quad_model = OLS(series, X_quad).fit()
        
        results['quadratic_trend'] = {
            'intercept': quad_model.params[0],
            'linear_coef': quad_model.params[1],
            'quadratic_coef': quad_model.params[2],
            'r_squared': quad_model.rsquared,
            'acceleration': 'decelerating' if quad_model.params[2] > 0 else 'accelerating'
        }
        
        # Period comparison
        mid_point = len(series) // 2
        results['period_comparison'] = {
            'early_period': f"{years[0]}-{years[mid_point-1]}",
            'late_period': f"{years[mid_point]}-{years[-1]}",
            'early_mean': series[:mid_point].mean(),
            'late_mean': series[mid_point:].mean(),
            'change': series[mid_point:].mean() - series[:mid_point].mean(),
            'percent_change': ((series[mid_point:].mean() - series[:mid_point].mean()) / series[:mid_point].mean()) * 100
        }
        
        return results
    
    def structural_break_detection(self, min_segment=3):
        """Detect structural breaks using Chow tests."""
        from statsmodels.regression.linear_model import OLS
        
        y = self.ts_df['wage_gap'].dropna().values
        years = self.ts_df['year'].values[:len(y)]
        n = len(y)
        X = sm.add_constant(np.arange(n))
        
        results = {'breakpoints': [], 'chow_tests': [], 'cusum': {}}
        
        # Sequential Chow tests
        chow_stats = []
        for t in range(min_segment, n - min_segment):
            y1, X1 = y[:t], X[:t]
            y2, X2 = y[t:], X[t:]
            
            rss_r = OLS(y, X).fit().ssr
            rss_u = OLS(y1, X1).fit().ssr + OLS(y2, X2).fit().ssr
            
            k = X.shape[1]
            chow_stat = ((rss_r - rss_u) / k) / (rss_u / (n - 2*k)) if rss_u > 0 else 0
            p_value = 1 - stats.f.cdf(chow_stat, k, n - 2*k)
            
            chow_stats.append({
                'break_year': int(years[t]),
                'break_index': t,
                'chow_statistic': chow_stat,
                'p_value': p_value,
                'significant': p_value < 0.05
            })
        
        results['chow_tests'] = chow_stats
        significant = [c for c in chow_stats if c['significant']]
        if significant:
            best = max(significant, key=lambda x: x['chow_statistic'])
            results['breakpoints'].append(best['break_year'])
        
        return results
    
    def granger_causality_analysis(self, max_lag=4):
        """Test Granger causality between male and female wages."""
        results = {'male_to_female': {}, 'female_to_male': {}}
        
        if 'male_wage' not in self.ts_df.columns:
            return results
        
        male = self.ts_df['male_wage'].dropna().values
        female = self.ts_df['female_wage'].dropna().values
        
        n = min(len(male), len(female))
        if n < max_lag + 3:
            return results
        
        data = np.column_stack([female[:n], male[:n]])
        
        try:
            gc_test = grangercausalitytests(data, maxlag=min(max_lag, n//3), verbose=False)
            for lag in gc_test:
                f_stat = gc_test[lag][0]['ssr_ftest'][0]
                p_val = gc_test[lag][0]['ssr_ftest'][1]
                results['male_to_female'][lag] = {'f_stat': f_stat, 'p_value': p_val, 'significant': p_val < 0.05}
        except:
            pass
        
        try:
            data_rev = np.column_stack([male[:n], female[:n]])
            gc_test = grangercausalitytests(data_rev, maxlag=min(max_lag, n//3), verbose=False)
            for lag in gc_test:
                f_stat = gc_test[lag][0]['ssr_ftest'][0]
                p_val = gc_test[lag][0]['ssr_ftest'][1]
                results['female_to_male'][lag] = {'f_stat': f_stat, 'p_value': p_val, 'significant': p_val < 0.05}
        except:
            pass
        
        return results
    
    def cointegration_analysis(self):
        """Test cointegration between male and female wages."""
        results = {}
        
        if 'male_wage' not in self.ts_df.columns:
            return results
        
        male = self.ts_df['male_wage'].dropna().values
        female = self.ts_df['female_wage'].dropna().values
        
        try:
            coint_stat, p_value, crit_values = coint(male, female)
            results['engle_granger'] = {
                'statistic': coint_stat,
                'p_value': p_value,
                'critical_values': {'1%': crit_values[0], '5%': crit_values[1], '10%': crit_values[2]},
                'cointegrated': p_value < 0.05,
                'interpretation': 'Male and female wages share a long-run equilibrium' if p_value < 0.05 
                                  else 'No cointegration detected'
            }
            
            # Estimate cointegrating relationship
            X = sm.add_constant(male)
            model = sm.OLS(female, X).fit()
            results['cointegrating_vector'] = {
                'coefficient': model.params[1],
                'interpretation': f"For every $1 increase in male wages, female wages increase by ${model.params[1]:.2f}"
            }
        except Exception as e:
            results['error'] = str(e)
        
        return results
    
    def var_model(self, max_lag=4):
        """Estimate Vector Autoregression model."""
        results = {}
        
        if 'male_wage' not in self.ts_df.columns:
            return results
        
        male = self.ts_df['male_wage'].dropna().values
        female = self.ts_df['female_wage'].dropna().values
        
        data = pd.DataFrame({'male_wage': male[:min(len(male), len(female))], 
                            'female_wage': female[:min(len(male), len(female))]})
        
        try:
            var_model = VAR(data)
            lag_order = var_model.select_order(maxlags=min(max_lag, len(data)//3))
            optimal_lag = lag_order.aic
            
            results['lag_selection'] = {'optimal_lag_aic': optimal_lag}
            
            fitted = var_model.fit(optimal_lag) if optimal_lag > 0 else var_model.fit(1)
            
            irf = fitted.irf(10)
            results['var_model'] = {
                'lag_order': fitted.k_ar,
                'irf_male_to_female': irf.irfs[:, 1, 0].tolist(),  # Response of female to male shock
                'irf_female_to_male': irf.irfs[:, 0, 1].tolist()   # Response of male to female shock
            }
        except Exception as e:
            results['error'] = str(e)
        
        return results
    
    def forecast_wage_gap(self, horizon=5):
        """Forecast future wage gaps using ARIMA."""
        from statsmodels.tsa.arima.model import ARIMA
        
        results = {}
        series = self.ts_df['wage_gap'].dropna()
        last_year = int(self.ts_df['year'].iloc[-1])
        
        try:
            # Fit ARIMA(1,1,1) - common choice for trending data
            model = ARIMA(series, order=(1, 1, 1))
            fitted = model.fit()
            
            forecast = fitted.get_forecast(steps=horizon)
            pred_mean = forecast.predicted_mean
            conf_int = forecast.conf_int(alpha=0.05)
            
            results['arima'] = {
                'order': (1, 1, 1),
                'aic': fitted.aic,
                'forecast_years': [last_year + i + 1 for i in range(horizon)],
                'point_forecast': pred_mean.tolist(),
                'lower_95': conf_int.iloc[:, 0].tolist(),
                'upper_95': conf_int.iloc[:, 1].tolist()
            }
            
            # Parity projection
            current_gap = float(series.iloc[-1])
            avg_annual_change = float((series.iloc[-1] - series.iloc[0]) / (len(series) - 1))
            years_to_parity = abs(current_gap / avg_annual_change) if avg_annual_change != 0 else float('inf')
            
            results['parity_projection'] = {
                'current_gap': current_gap,
                'avg_annual_change': avg_annual_change,
                'years_to_parity': years_to_parity,
                'projected_parity_year': last_year + int(years_to_parity) if years_to_parity != float('inf') else None
            }
        except Exception as e:
            results['error'] = str(e)
        
        return results

# Initialize analyzer with REAL data
analyzer = YearlyWageAnalyzer(ts_data)
print(f"✅ Time Series Analyzer initialized with REAL Statistics Canada data")
print(f"   Data range: {analyzer.data['year'].min()} - {analyzer.data['year'].max()}")
print(f"   Observations: {len(analyzer.data)} years")
print(f"   Source: {ts_df['source'].iloc[0] if 'source' in ts_df.columns else 'Statistics Canada'}")

## 2. Visualize Wage Trends Over Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if 'male_wage' in ts_df.columns and 'female_wage' in ts_df.columns:
    # 1. Male vs Female Wages
    ax1 = axes[0, 0]
    ax1.plot(ts_df['year'], ts_df['male_wage'], 'b-o', label='Male', linewidth=2, markersize=4)
    ax1.plot(ts_df['year'], ts_df['female_wage'], 'r-o', label='Female', linewidth=2, markersize=4)
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Hourly Wage ($)')
    ax1.set_title('Average Hourly Wages by Gender')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. Wage Gap Over Time
    ax2 = axes[0, 1]
    ax2.fill_between(ts_df['year'], ts_df['wage_gap'], alpha=0.3, color='purple')
    ax2.plot(ts_df['year'], ts_df['wage_gap'], 'purple', linewidth=2)
    ax2.axhline(y=0, color='green', linestyle='--', label='Parity')
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Wage Gap (%)')
    ax2.set_title('Gender Wage Gap Over Time')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # 3. Wage Ratio
    ax3 = axes[1, 0]
    ax3.plot(ts_df['year'], ts_df['wage_ratio'] * 100, 'g-o', linewidth=2, markersize=4)
    ax3.axhline(y=100, color='red', linestyle='--', label='Full Parity')
    ax3.set_xlabel('Year')
    ax3.set_ylabel('Female/Male Ratio (%)')
    ax3.set_title('Female-to-Male Wage Ratio')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Year-over-Year Change
    ax4 = axes[1, 1]
    gap_change = ts_df['wage_gap'].diff()
    colors = ['green' if x < 0 else 'red' for x in gap_change]
    ax4.bar(ts_df['year'][1:], gap_change[1:], color=colors[1:], alpha=0.7)
    ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax4.set_xlabel('Year')
    ax4.set_ylabel('Change in Gap (pp)')
    ax4.set_title('Year-over-Year Change in Wage Gap')
    ax4.grid(True, alpha=0.3)

else:
    # Fallback visualization
    axes[0, 0].text(0.5, 0.5, 'Male/Female wage columns not available', ha='center', va='center')

plt.tight_layout()
plt.savefig(str(Path.cwd().parent / 'reports' / 'wage_trends_over_time.png'), dpi=150)
plt.show()
print("\n✅ Saved: reports/wage_trends_over_time.png")

## 3. Unit Root Tests (Stationarity Analysis)

Testing whether the wage gap series is stationary:
- **ADF Test**: H₀ = unit root exists (non-stationary)
- **KPSS Test**: H₀ = series is stationary

In [ ]:
# Advanced Unit Root Tests - Scientific Standards
from src.statistical_tests import AdvancedStatisticalTests

tests = AdvancedStatisticalTests(significance_level=0.05)

print("="*70)
print("COMPREHENSIVE UNIT ROOT TESTING (Scientific Standards)")
print("="*70)

# Test the nominal wage gap
wage_gap_series = ts_df['wage_gap'].dropna()

# Confirmatory strategy: ADF + KPSS
unit_root_results = tests.confirmatory_unit_root(wage_gap_series)

print("\n📊 NOMINAL WAGE GAP STATIONARITY TESTS:")
print("-"*60)

for test_name, result in unit_root_results.items():
    if test_name != 'interpretation':
        print(f"\n{result.test_name}:")
        print(f"   Test Statistic: {result.statistic:.4f}")
        print(f"   P-value: {result.p_value:.4f}" if not np.isnan(result.p_value) else "   P-value: N/A")
        if result.critical_values:
            print("   Critical Values:")
            for level, cv in list(result.critical_values.items())[:3]:
                print(f"      {level}: {cv:.4f}")
        status = "✓ REJECT H₀" if result.reject_null else "✗ FAIL TO REJECT H₀"
        print(f"   Decision (α=0.05): {status}")
        print(f"   Conclusion: {result.conclusion}")

print(f"\n🎯 OVERALL INTERPRETATION: {unit_root_results['interpretation']}")

# Test real wage gap too
print("\n" + "="*70)
print("📊 REAL WAGE GAP (Inflation-Adjusted) STATIONARITY:")
print("-"*60)
real_gap = ts_df['real_wage_gap'].dropna()
adf_real = tests.adf_test(real_gap)
print(f"ADF Statistic: {adf_real.statistic:.4f}, p-value: {adf_real.p_value:.4f}")
print(f"Conclusion: {adf_real.conclusion}")

## 4. Trend Analysis

In [ ]:
trend_results = analyzer.trend_analysis()

print("="*60)
print("TREND ANALYSIS")
print("="*60)

if 'linear_trend' in trend_results:
    lt = trend_results['linear_trend']
    print(f"\n📈 Linear Trend:")
    print(f"   Slope: {lt['slope']:.4f} percentage points per year")
    print(f"   R²: {lt['r_squared']:.4f}")
    print(f"   P-value: {lt['p_value_slope']:.4f}")
    print(f"   Significant: {'Yes ✓' if lt['significant_trend'] else 'No'}")
    print(f"\n   Interpretation: {lt['slope_interpretation']}")

if 'quadratic_trend' in trend_results:
    qt = trend_results['quadratic_trend']
    print(f"\n📉 Quadratic Trend:")
    print(f"   Linear coefficient: {qt['linear_coef']:.4f}")
    print(f"   Quadratic coefficient: {qt['quadratic_coef']:.6f}")
    print(f"   R²: {qt['r_squared']:.4f}")
    print(f"   Trend is {qt['acceleration']}")

if 'period_comparison' in trend_results:
    pc = trend_results['period_comparison']
    print(f"\n📊 Period Comparison:")
    print(f"   Early period ({pc['early_period']}): {pc['early_mean']:.2f}%")
    print(f"   Late period ({pc['late_period']}): {pc['late_mean']:.2f}%")
    print(f"   Change: {pc['change']:.2f} pp ({pc['percent_change']:.1f}%)")

In [ ]:
# Visualize trend
if 'wage_gap' in ts_df.columns and 'linear_trend' in trend_results:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    years = ts_df['year'].values
    gap = ts_df['wage_gap'].values
    
    # Actual data
    ax.scatter(years, gap, color='purple', s=60, zorder=5, label='Observed')
    
    # Linear trend
    lt = trend_results['linear_trend']
    trend_line = lt['intercept'] + lt['slope'] * years
    ax.plot(years, trend_line, 'b--', linewidth=2, label=f'Linear Trend (slope={lt["slope"]:.3f})')
    
    # Quadratic trend
    if 'quadratic_trend' in trend_results:
        qt = trend_results['quadratic_trend']
        quad_line = qt['intercept'] + qt['linear_coef'] * years + qt['quadratic_coef'] * years**2
        ax.plot(years, quad_line, 'r-', linewidth=2, label='Quadratic Trend')
    
    ax.axhline(y=0, color='green', linestyle=':', alpha=0.7, label='Parity')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Wage Gap (%)', fontsize=12)
    ax.set_title('Gender Wage Gap Trend Analysis', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(Path.cwd().parent / 'reports' / 'wage_gap_trend.png'), dpi=150)
    plt.show()

## 5. Structural Break Detection

Identifying when significant changes occurred in wage gap dynamics (e.g., policy interventions).

In [ ]:
break_results = analyzer.structural_break_detection()

print("="*60)
print("STRUCTURAL BREAK DETECTION")
print("="*60)

# CUSUM Test
if 'cusum' in break_results and break_results['cusum']:
    cusum = break_results['cusum']
    print(f"\n📊 CUSUM Test:")
    print(f"   Max CUSUM statistic: {cusum['statistic']:.4f}")
    print(f"   Critical value (5%): {cusum['critical_value']:.4f}")
    print(f"   Structural break detected: {'Yes ⚠️' if cusum['structural_break_detected'] else 'No'}")

# Multiple breaks
if 'multiple_breaks' in break_results and break_results['multiple_breaks']:
    mb = break_results['multiple_breaks']
    print(f"\n📈 Multiple Breakpoint Analysis (Bai-Perron style):")
    print(f"   No break RSS: {mb['no_break_rss']:.4f}")
    if mb.get('one_break'):
        print(f"   One break RSS: {mb['one_break']['rss']:.4f} (year: {mb['one_break']['year']})")
    if mb.get('two_breaks'):
        print(f"   Two breaks RSS: {mb['two_breaks']['rss']:.4f} (years: {mb['two_breaks']['years']})")

# Identified breakpoints
if break_results.get('breakpoints'):
    print(f"\n🎯 Significant Breakpoints Detected: {break_results['breakpoints']}")

In [ ]:
# Visualize Chow test statistics
if 'chow_tests' in break_results and break_results['chow_tests']:
    chow_df = pd.DataFrame(break_results['chow_tests'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Chow statistics
    ax1 = axes[0]
    ax1.plot(chow_df['break_year'], chow_df['chow_statistic'], 'b-', linewidth=2)
    ax1.fill_between(chow_df['break_year'], chow_df['chow_statistic'], alpha=0.3)
    
    # Mark significant breaks
    sig_breaks = chow_df[chow_df['significant']]
    if len(sig_breaks) > 0:
        ax1.scatter(sig_breaks['break_year'], sig_breaks['chow_statistic'], 
                   color='red', s=100, zorder=5, label='Significant (p<0.05)')
    
    ax1.set_xlabel('Potential Break Year')
    ax1.set_ylabel('Chow Statistic')
    ax1.set_title('Sequential Chow Test for Structural Breaks')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # P-values
    ax2 = axes[1]
    ax2.semilogy(chow_df['break_year'], chow_df['p_value'], 'g-', linewidth=2)
    ax2.axhline(y=0.05, color='red', linestyle='--', label='5% significance')
    ax2.axhline(y=0.01, color='orange', linestyle='--', label='1% significance')
    ax2.set_xlabel('Potential Break Year')
    ax2.set_ylabel('P-value (log scale)')
    ax2.set_title('Chow Test P-values')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(Path.cwd().parent / 'reports' / 'structural_breaks.png'), dpi=150)
    plt.show()

## 6. Granger Causality Analysis

Testing whether changes in male wages predict future changes in female wages, and vice versa.

In [ ]:
granger_results = analyzer.granger_causality_analysis(max_lag=4)

print("="*60)
print("GRANGER CAUSALITY TESTS")
print("="*60)

if 'male_to_female' in granger_results:
    print("\n📊 Male Wages → Female Wages (Do male wages predict female wages?):")
    print("-" * 50)
    for lag, stats in granger_results['male_to_female'].items():
        sig = '✓' if stats['significant'] else ''
        print(f"   Lag {lag}: F-stat = {stats['f_stat']:.3f}, p-value = {stats['p_value']:.4f} {sig}")

if 'female_to_male' in granger_results:
    print("\n📊 Female Wages → Male Wages (Do female wages predict male wages?):")
    print("-" * 50)
    for lag, stats in granger_results['female_to_male'].items():
        sig = '✓' if stats['significant'] else ''
        print(f"   Lag {lag}: F-stat = {stats['f_stat']:.3f}, p-value = {stats['p_value']:.4f} {sig}")

# Interpretation
print("\n🔍 Interpretation:")
if granger_results:
    m2f_sig = any(v['significant'] for v in granger_results.get('male_to_female', {}).values())
    f2m_sig = any(v['significant'] for v in granger_results.get('female_to_male', {}).values())
    
    if m2f_sig and not f2m_sig:
        print("   Male wages Granger-cause female wages (unidirectional)")
        print("   → Male wage changes precede female wage changes")
    elif f2m_sig and not m2f_sig:
        print("   Female wages Granger-cause male wages (unidirectional)")
    elif m2f_sig and f2m_sig:
        print("   Bidirectional Granger causality")
        print("   → Male and female wages influence each other")
    else:
        print("   No significant Granger causality detected")

## 7. Cointegration Analysis

Testing whether male and female wages share a long-run equilibrium relationship.

In [ ]:
coint_results = analyzer.cointegration_analysis()

print("="*60)
print("COINTEGRATION ANALYSIS")
print("="*60)

if 'engle_granger' in coint_results:
    eg = coint_results['engle_granger']
    print("\n📊 Engle-Granger Two-Step Cointegration Test:")
    print("-" * 50)
    print(f"   Test Statistic: {eg['statistic']:.4f}")
    print(f"   P-value: {eg['p_value']:.4f}")
    print(f"   Critical Values:")
    for level, cv in eg['critical_values'].items():
        print(f"      {level}: {cv:.4f}")
    print(f"\n   Cointegrated: {'Yes ✓' if eg['cointegrated'] else 'No'}")
    print(f"   {eg['interpretation']}")

if 'cointegrating_vector' in coint_results:
    cv = coint_results['cointegrating_vector']
    print(f"\n📈 Cointegrating Relationship:")
    print(f"   {cv['interpretation']}")
    print(f"\n   Implication: For every $1 increase in male wages,")
    print(f"   female wages increase by ${cv['coefficient']:.2f} in the long run")

## 8. Vector Autoregression (VAR) Model

Capturing dynamic interdependencies between male and female wages.

In [ ]:
var_results = analyzer.var_model(max_lag=4)

print("="*60)
print("VECTOR AUTOREGRESSION MODEL")
print("="*60)

if 'lag_selection' in var_results:
    ls = var_results['lag_selection']
    print(f"\n📊 Optimal Lag Selection: {ls['optimal_lag_aic']} (by AIC)")

if 'var_model' in var_results:
    vm = var_results['var_model']
    print(f"\n📈 VAR({vm['lag_order']}) Model Estimated")
    
    # Impulse Response Analysis
    print("\n🎯 Impulse Response Functions (10-period horizon):")
    print("   Response of female wage to male wage shock:")
    for i, irf in enumerate(vm['irf_male_to_female'][:5]):
        print(f"      Period {i}: {irf:.4f}")
    
    print("\n   Response of male wage to female wage shock:")
    for i, irf in enumerate(vm['irf_female_to_male'][:5]):
        print(f"      Period {i}: {irf:.4f}")

In [ ]:
# Visualize Impulse Response Functions
if 'var_model' in var_results:
    vm = var_results['var_model']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    periods = range(len(vm['irf_male_to_female']))
    
    # Male to Female IRF
    ax1 = axes[0]
    ax1.bar(periods, vm['irf_male_to_female'], color='blue', alpha=0.7)
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Periods After Shock')
    ax1.set_ylabel('Response')
    ax1.set_title('Response of Female Wage to Male Wage Shock')
    ax1.grid(True, alpha=0.3)
    
    # Female to Male IRF
    ax2 = axes[1]
    ax2.bar(periods, vm['irf_female_to_male'], color='red', alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Periods After Shock')
    ax2.set_ylabel('Response')
    ax2.set_title('Response of Male Wage to Female Wage Shock')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(Path.cwd().parent / 'reports' / 'impulse_response.png'), dpi=150)
    plt.show()

## 9. Forecasting Future Wage Gap

In [ ]:
forecast_results = analyzer.forecast_wage_gap(horizon=10)

print("="*60)
print("WAGE GAP FORECASTING")
print("="*60)

if 'arima' in forecast_results:
    arima = forecast_results['arima']
    print(f"\n📊 ARIMA{arima['order']} Model (AIC: {arima['aic']:.2f})")
    print("\n📈 Wage Gap Forecasts:")
    print("-" * 50)
    print(f"{'Year':<10} {'Forecast':<12} {'95% CI':>20}")
    print("-" * 50)
    
    for i, year in enumerate(arima['forecast_years']):
        forecast = arima['point_forecast'][i]
        lower = arima['lower_95'][i]
        upper = arima['upper_95'][i]
        print(f"{year:<10} {forecast:>8.2f}%    [{lower:>6.2f}%, {upper:>6.2f}%]")

if 'parity_projection' in forecast_results:
    pp = forecast_results['parity_projection']
    print(f"\n🎯 Parity Projection:")
    print(f"   Current gap: {pp['current_gap']:.2f}%")
    print(f"   Annual decrease: {pp['annual_decrease']:.3f} pp")
    print(f"   Years to parity: {pp['years_to_parity']}")
    print(f"   Projected parity year: {pp['projected_parity_year']}")

In [ ]:
# Visualize forecasts
if 'arima' in forecast_results and 'wage_gap' in ts_df.columns:
    arima = forecast_results['arima']
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Historical data
    ax.plot(ts_df['year'], ts_df['wage_gap'], 'b-o', linewidth=2, 
            markersize=5, label='Historical')
    
    # Forecasts
    forecast_years = arima['forecast_years']
    ax.plot(forecast_years, arima['point_forecast'], 'r--o', linewidth=2,
            markersize=5, label='Forecast')
    
    # Confidence interval
    ax.fill_between(forecast_years, arima['lower_95'], arima['upper_95'],
                    alpha=0.2, color='red', label='95% CI')
    
    # Parity line
    ax.axhline(y=0, color='green', linestyle=':', linewidth=2, label='Parity')
    
    # Projected parity year
    if 'parity_projection' in forecast_results:
        pp = forecast_results['parity_projection']
        ax.axvline(x=pp['projected_parity_year'], color='purple', linestyle='--',
                   alpha=0.7, label=f"Projected Parity ({pp['projected_parity_year']})")
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Wage Gap (%)', fontsize=12)
    ax.set_title('Gender Wage Gap: Historical Trend and Forecast', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(Path.cwd().parent / 'reports' / 'wage_gap_forecast.png'), dpi=150)
    plt.show()

## 10. Comprehensive Summary

In [ ]:
# Macro-Augmented Wage Gap Regression Analysis
import statsmodels.api as sm
from src.statistical_tests import AdvancedStatisticalTests, create_diagnostic_summary_table

print("="*70)
print("MACRO-AUGMENTED WAGE GAP REGRESSION ANALYSIS")
print("="*70)

# Prepare regression data
reg_data = ts_df.dropna().copy()

# Model 1: Simple trend
X1 = sm.add_constant(reg_data['trend'])
model1 = sm.OLS(reg_data['wage_gap'], X1).fit()

print("\n📊 Model 1: Simple Time Trend")
print("-"*50)
print(f"   Wage Gap = {model1.params['const']:.3f} + ({model1.params['trend']:.4f}) × Trend")
print(f"   R² = {model1.rsquared:.4f}")
print(f"   Trend coefficient: {model1.params['trend']:.4f} (p={model1.pvalues['trend']:.4f})")
print(f"   → Wage gap {'decreases' if model1.params['trend'] < 0 else 'increases'} by {abs(model1.params['trend']):.3f} pp/year")

# Model 2: With macro controls
X2 = sm.add_constant(reg_data[['trend', 'unemployment', 'gdp_growth', 'covid']])
model2 = sm.OLS(reg_data['wage_gap'], X2).fit()

print("\n📊 Model 2: With Macroeconomic Controls")
print("-"*50)
print(model2.summary().tables[1])

# Model 3: Real wage gap with macro controls
X3 = sm.add_constant(reg_data[['trend', 'unemployment', 'inflation', 'covid']])
model3 = sm.OLS(reg_data['real_wage_gap'], X3).fit()

print("\n📊 Model 3: Real Wage Gap (Inflation-Adjusted)")
print("-"*50)
print(model3.summary().tables[1])

# Diagnostic tests for preferred model
print("\n" + "="*70)
print("REGRESSION DIAGNOSTICS (Model 2)")
print("="*70)

tests = AdvancedStatisticalTests()
diagnostics = tests.run_regression_diagnostics(model2)
report = tests.print_diagnostic_report(diagnostics)
print(report)

In [ ]:
# Save results
import json

# Convert results to JSON-serializable format
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif hasattr(obj, '__dict__'):
        return {k: make_serializable(v) for k, v in obj.__dict__.items()}
    else:
        return obj

serializable_results = make_serializable(full_results)

output_path = Path.cwd().parent / 'reports' / 'time_series_analysis_results.json'
with open(output_path, 'w') as f:
    json.dump(serializable_results, f, indent=2, default=str)

print(f"✅ Results saved to: {output_path}")

## Research-Grade Time Series Summary

This notebook applied **advanced time series econometrics** to analyze the temporal dynamics of Canada's gender wage gap (2010-2025).

### Methods Applied

| Method | Purpose | Key Finding | Citation |
|--------|---------|-------------|----------|
| **ADF Test** | Unit root detection | Tests gap convergence | Dickey & Fuller (1979) |
| **KPSS Test** | Stationarity confirmation | Validates ADF | Kwiatkowski et al. (1992) |
| **Zivot-Andrews** | Endogenous break point | Identifies policy impacts | Zivot & Andrews (1992) |
| **Phillips Curve** | Macro-gap relationship | Counter-cyclical dynamics | Phillips (1958) |
| **Cointegration** | Long-run equilibrium | Gap-unemployment link | Engle & Granger (1987) |
| **Granger Causality** | Predictive relationships | Lead-lag dynamics | Granger (1969) |
| **ARIMA** | Forecasting | Gap trajectory | Box & Jenkins (1976) |

### Key Findings Summary

```
┌────────────────────────────────────────────────────────────────────────┐
│ TIME SERIES EVIDENCE SYNTHESIS                                         │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│ 1. TREND: Linear trend shows [narrowing/widening] gap over 2010-2025  │
│                                                                        │
│ 2. STATIONARITY:                                                       │
│    • ADF + KPSS confirmatory testing                                  │
│    • Implication for mean reversion                                    │
│                                                                        │
│ 3. STRUCTURAL BREAKS:                                                  │
│    • COVID-19 (2020): [Significant/Not significant] break             │
│    • Pay Equity Act (2018): Policy evaluation                          │
│                                                                        │
│ 4. MACRO RELATIONSHIPS:                                                │
│    • Phillips curve: β = [value] (counter/pro-cyclical)               │
│    • Granger causality: Unemployment → Gap dynamics                   │
│                                                                        │
│ 5. FORECAST: Projected gap trajectory to 2030                          │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

### Policy Implications

| Finding | Policy Recommendation | Mechanism |
|---------|----------------------|-----------|
| Gap counter-cyclical | Automatic stabilizers | Protect during recessions |
| Structural break at policy | Continue effective policies | Evidence-based continuation |
| Long-run cointegration | Target equilibrium | Structural interventions |
| Slow convergence | Accelerate progress | Proactive measures needed |

### Limitations & Future Research

1. **Sample Size**: 16 annual observations limits power of time series tests
2. **Aggregation**: Annual data may mask within-year dynamics
3. **Identification**: Macro correlations ≠ causation
4. **External Validity**: Canadian-specific findings

### References
- Box, G. E., & Jenkins, G. M. (1976). *Time Series Analysis*. Holden-Day.
- Dickey, D. A., & Fuller, W. A. (1979). Distribution of estimators. *JASA*.
- Engle, R. F., & Granger, C. W. (1987). Co-integration and error correction. *Econometrica*.
- Granger, C. W. (1969). Investigating causal relations. *Econometrica*.
- Kwiatkowski, D., et al. (1992). Testing stationarity against unit root. *JE*.
- Phillips, A. W. (1958). The relation between unemployment and the rate of change of money wage rates. *Economica*.
- Zivot, E., & Andrews, D. W. K. (1992). Further evidence on the great crash. *JBES*.

---

## 10. Geographic Time Series: Animated Provincial Wage Gap Map

Visualize how the gender wage gap has evolved **geographically** across Canadian provinces from 2010-2025 using an animated choropleth map.

This combines our time series analysis with spatial visualization to reveal:
- Which provinces have made the most progress
- Regional patterns in wage gap reduction
- East-West disparities over time

In [ ]:
# ============================================================================
# ANIMATED CHOROPLETH MAP: Provincial Wage Gap Evolution
# ============================================================================
import plotly.express as px

print("=" * 70)
print("GEOGRAPHIC TIME SERIES: ANIMATED PROVINCIAL WAGE GAP MAP")
print("=" * 70)

# Province code to ISO 3166-2 mapping for Plotly
PROV_TO_ISO = {
    10: 'CA-NL',  # Newfoundland and Labrador
    11: 'CA-PE',  # Prince Edward Island
    12: 'CA-NS',  # Nova Scotia
    13: 'CA-NB',  # New Brunswick
    24: 'CA-QC',  # Quebec
    35: 'CA-ON',  # Ontario
    46: 'CA-MB',  # Manitoba
    47: 'CA-SK',  # Saskatchewan
    48: 'CA-AB',  # Alberta
    59: 'CA-BC',  # British Columbia
}

PROV_NAMES = {
    10: 'Newfoundland & Labrador',
    11: 'Prince Edward Island',
    12: 'Nova Scotia',
    13: 'New Brunswick',
    24: 'Quebec',
    35: 'Ontario',
    46: 'Manitoba',
    47: 'Saskatchewan',
    48: 'Alberta',
    59: 'British Columbia',
}

# Calculate wage gap by province and year
prov_col = COLS.PROV if COLS.PROV in df.columns else 'PROV'

provincial_gap_data = []

for year in sorted(df[year_col].unique()):
    for prov_code in PROV_TO_ISO.keys():
        prov_year_data = df[(df[year_col] == year) & (df[prov_col] == prov_code)]
        
        if len(prov_year_data) > 50:  # Minimum sample size
            male_wages = prov_year_data[prov_year_data['IS_FEMALE'] == 0][wage_col]
            female_wages = prov_year_data[prov_year_data['IS_FEMALE'] == 1][wage_col]
            
            if len(male_wages) > 10 and len(female_wages) > 10:
                male_mean = male_wages.mean()
                female_mean = female_wages.mean()
                gap_pct = (male_mean - female_mean) / male_mean * 100 if male_mean > 0 else 0
                
                provincial_gap_data.append({
                    'year': int(year),
                    'prov_code': prov_code,
                    'iso_code': PROV_TO_ISO[prov_code],
                    'province': PROV_NAMES[prov_code],
                    'male_wage': male_mean,
                    'female_wage': female_mean,
                    'wage_gap_pct': gap_pct,
                    'n_male': len(male_wages),
                    'n_female': len(female_wages)
                })

prov_gap_df = pd.DataFrame(provincial_gap_data)
print(f"✓ Calculated wage gaps for {len(prov_gap_df)} province-year combinations")
print(f"  Years: {prov_gap_df['year'].min()} - {prov_gap_df['year'].max()}")
print(f"  Provinces: {prov_gap_df['province'].nunique()}")

In [ ]:
# ============================================================================
# ANIMATED CHOROPLETH MAP
# ============================================================================

# Create animated choropleth
fig = px.choropleth(
    prov_gap_df,
    locations='iso_code',
    locationmode='ISO-3166-2',
    color='wage_gap_pct',
    hover_name='province',
    hover_data={
        'iso_code': False,
        'wage_gap_pct': ':.1f',
        'male_wage': ':$.2f',
        'female_wage': ':$.2f',
        'n_male': ':,',
        'n_female': ':,'
    },
    animation_frame='year',
    color_continuous_scale='RdYlGn_r',  # Red = high gap, Green = low gap
    range_color=[5, 20],  # Typical gap range
    scope='north america',
    title='<b>Gender Wage Gap by Province (2010-2025)</b><br><sup>Real wages in 2010 constant dollars</sup>',
    labels={
        'wage_gap_pct': 'Wage Gap (%)',
        'male_wage': 'Male Avg ($)',
        'female_wage': 'Female Avg ($)',
        'n_male': 'Male n',
        'n_female': 'Female n'
    }
)

# Focus on Canada
fig.update_geos(
    visible=True,
    resolution=50,
    showcountries=True,
    countrycolor='lightgray',
    showsubunits=True,
    subunitcolor='white',
    showlakes=True,
    lakecolor='lightblue',
    center=dict(lat=60, lon=-95),
    projection_scale=2.5,
    lataxis_range=[42, 72],
    lonaxis_range=[-141, -52]
)

fig.update_layout(
    height=600,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(
        title='Wage Gap (%)',
        ticksuffix='%'
    )
)

# Save as interactive HTML
fig.write_html('../reports/figures/provincial_wage_gap_animated.html')
print("📊 Interactive map saved: reports/figures/provincial_wage_gap_animated.html")

fig.show()

In [ ]:
# ============================================================================
# PROVINCIAL TREND COMPARISON: Small Multiples
# ============================================================================

# Line chart showing gap evolution by province
fig, ax = plt.subplots(figsize=(14, 8))

# Color palette for provinces
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, prov_code in enumerate(sorted(PROV_TO_ISO.keys())):
    prov_data = prov_gap_df[prov_gap_df['prov_code'] == prov_code].sort_values('year')
    if len(prov_data) > 0:
        ax.plot(prov_data['year'], prov_data['wage_gap_pct'], 
                marker='o', markersize=4, linewidth=2,
                label=PROV_NAMES[prov_code], color=colors[i], alpha=0.8)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Gender Wage Gap (%)', fontsize=12)
ax.set_title('Gender Wage Gap Trend by Province (2010-2025)\nReal Wages in 2010 Constant Dollars', 
             fontsize=14, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Parity')

# Add shaded regions for economic periods
ax.axvspan(2020, 2021, alpha=0.15, color='red', label='COVID-19')

plt.tight_layout()
plt.savefig('../reports/figures/provincial_gap_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Provincial trends saved: reports/figures/provincial_gap_trends.png")

# Summary statistics by province
print("\n" + "=" * 70)
print("PROVINCIAL WAGE GAP SUMMARY (2010-2025 Average)")
print("=" * 70)

summary = prov_gap_df.groupby('province').agg({
    'wage_gap_pct': ['mean', 'std', 'min', 'max'],
    'year': 'count'
}).round(2)
summary.columns = ['Avg Gap (%)', 'Std Dev', 'Min Gap', 'Max Gap', 'Years']
summary = summary.sort_values('Avg Gap (%)', ascending=False)
print(summary)